In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

/home/arjunverma/miniconda3/envs/common-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [4]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [6]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [7]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])



The average distance from the Earth to the Moon is **384,400 kilometers (238,900 miles)**.

It's important to note that this is an **average**. The Moon's orbit around the Earth is not a perfect circle, but rather an ellipse. This means the distance varies:

*   **Perigee:** The closest point in the Moon's orbit to Earth, which is about 363,300 km (225,700 miles).
*   **Apogee:** The farthest point in the Moon's orbit from Earth, which is about 405,500 km (251,900 miles).


In [8]:
model.invoke('How far is moon from the earth?').content

"The average distance between the Earth and the Moon is about **384,400 kilometers (238,900 miles)**.\n\nIt's important to note that this is an average because the Moon's orbit around the Earth is not a perfect circle; it's an ellipse. This means the distance varies:\n\n*   **Perigee:** The closest point in the Moon's orbit to Earth, approximately 363,300 km (225,700 miles).\n*   **Apogee:** The farthest point in the Moon's orbit from Earth, approximately 405,500 km (251,900 miles)."